In [ ]:
# Forecasting of products demand based on M5 dataset. Step 1 - Baseline and reseacrh

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor


# Folder of data
DATA_DIR = Path("data/m5")

# Main initial dataframes download
sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv")
prices = pd.read_csv(DATA_DIR / "sell_prices.csv")


#### Preparing dataframes for further processing
sales = sales.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name="day",value_name = 'sales'
)
sales['day'] = sales['day'].str.split('_').str[1].astype('int64')

calendar['date'] = pd.to_datetime(calendar['date'])
calendar['day'] = range(1, len(calendar) + 1)
s = calendar['day'] - calendar['day'].shift(1)
if (s.max(skipna = True)==1) and (s.min(skipna = True)==1):
     print('calendar describes every day')
else:
    print('missing dates in calendar')

# We don't know the real date of product release in a specific store, but we think that it is the beginning of the first week when we see the price

week_start_sale = pd.DataFrame(prices.groupby(['store_id','item_id'])['wm_yr_wk'].min().reset_index().sort_values(by='wm_yr_wk'))
week_first_day = calendar.groupby(by ='wm_yr_wk')['date'].min().reset_index().rename(columns={'date':'start_sale'})
start_sale = week_start_sale.merge(week_first_day, how = 'left', on='wm_yr_wk').drop(columns = ['wm_yr_wk'])

calendar describes every day


In [3]:
## Enrichment of sales data by Calendar, Days after release, Features of lag and rolling
sales1 = sales.merge(calendar, how = 'right', on='day')
sales1['snap'] = np.where(((sales1['state_id']=='CA') & (sales1['snap_CA']==1)) | ((sales1['state_id']=='WI') & (sales1['snap_WI']==1)) | ((sales1['state_id']=='TX') & (sales1['snap_TX']==1)),1,0)
sales1 = sales1.drop(columns=['snap_TX','snap_WI','snap_CA'])
sales2 = sales1.merge(prices, how='left', on=['item_id','store_id','wm_yr_wk'])

# diff2 = (sales2['date'] - sales2.groupby(['item_id','store_id'])['date'].shift(1)).dt.days
sales3 = sales2.merge(start_sale, how='left', on=['store_id','item_id'])

## Preparing with features
sales3['days_after_release'] = (sales3['date'] - sales3['start_sale']).dt.days
sales3 = sales3[sales3['date']>sales3['start_sale']].drop(columns=['start_sale','wm_yr_wk','weekday']) ## filtering all the data before release of the product. just in case
sales3['week_of_year'] = sales3.date.dt.isocalendar().week
sales3['day_of_month'] = sales3['date'].dt.day

g = sales3.groupby(['item_id', 'store_id'])

sales3['lag1']=g['sales'].shift(1)
sales3['lag7']=g['sales'].shift(7)
sales3['lag28']=g['sales'].shift(28)
sales3['rolling_mean7'] = g['sales'].transform(lambda x: x.shift(1).rolling(7).mean())
sales3['rolling_mean28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).mean())
sales3['rolling_median28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).median())
sales3['rolling_std7'] = g['sales'].transform(lambda x: x.shift(1).rolling(7).std())
sales3['rolling_std28'] = g['sales'].transform(lambda x: x.shift(1).rolling(28).std())

sales3['price1'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(7).median()))
sales3['price7'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(7).median()))
sales3['price28'] = (g['sell_price'].transform(lambda x: x.shift(1).rolling(28).median()))

sales3['trend'] = sales3['rolling_mean7']/sales3['rolling_mean28']
sales3['discount1'] = sales3['sell_price']/sales3['price1']
sales3['discount7'] = sales3['sell_price']/sales3['price7']
sales3['discount28'] = sales3['sell_price']/sales3['price28']
# sales3 = sales3[sales3['sales']>0]

diff3 = (sales3['date'] - sales3.groupby(['item_id','store_id'])['date'].shift(1)).dt.days
if diff3.min(skipna=True)==1 and diff3.max(skipna=True)==1:
    print('We are ready to go further')
else:
    print('Stop and check. Now you have gaps in the product calendar of repeated dates')

We are ready to go further


## We finished with features taken into account. Starting to build seasonal naive baseline

In [4]:
## At first we separate train, validation and test
df_regular =sales3.sort_values(by = ['item_id','store_id','date']).copy()

test_start = df_regular['date'].max() - pd.Timedelta(days=27)
test_df = df_regular[df_regular['date']>=test_start].copy() ## later that test_start

train_df = df_regular[df_regular['date']<test_start].copy() ## before test_start

validation_start = test_start - pd.Timedelta(days=27) ## cutoff for validation
validation_df = train_df[train_df['date'] >= validation_start].copy()

train_window = train_df[
    (train_df['date'] >= validation_start - pd.Timedelta(days=179)) & (train_df['date']<validation_start)].copy() ## before validation start but half a year period due to limitation of memory

## Here we determine what features are categorical and numerical, clear NaN in categorical features

cat_features = [
    'item_id', 'dept_id','store_id',
    'event_name_1', 'event_type_1',
]
num_features = [
    'wday','month','year','week_of_year','day_of_month','snap',
    'sell_price','days_after_release',
    'lag1', 'lag7', 'lag28',
    'rolling_mean7', 'rolling_mean28','rolling_median28',
    'rolling_std7','rolling_std28',
    'price1','price7','price28',
    'trend',
    'discount1','discount7','discount28',
    'horizon'
]
feature_cols = cat_features + num_features

In [5]:
### Baseline - seasonal forecast
def seasonal_naive_forecast(
    train_df,
    horizon=28,
    season_length=7
):
    """
    Seasonal Naive forecast для одной временной серии.
    train_df: DataFrame с колонками date и sales

    horizon: горизонт прогноза в днях.
    season_length: длина сезонности. Для недельной сезонности = 7.
    """
    if train_df['item_id'].nunique()==1 and train_df['store_id'].nunique()==1:
        item_id = train_df['item_id'].unique()[0]
        store_id = train_df['store_id'].unique()[0]
        train_df = train_df.sort_values(by ='date' ).copy()
        diff = (train_df['date'] - train_df['date'].shift(1)).dt.days
        last_date = train_df['date'].iloc[-1]
        future_dates = pd.date_range(start=last_date+pd.Timedelta(days = 1), end=last_date + pd.Timedelta(days=horizon))

        if diff.max(skipna=True)==1 and diff.min(skipna=True)==1:

            df = pd.DataFrame([])
            s = pd.Series([])
            df['forecast_dates'] = future_dates
            df['item_id'] = item_id
            df['store_id'] = store_id
            if season_length < horizon:
                while len(s)< horizon:
                    s = pd.concat([s, train_df.tail(season_length)['sales']])
                df['forecast_sales'] = s[0:horizon].reset_index(drop=True)

            elif season_length >= horizon:
                df['forecast_sales'] = train_df.tail(season_length )['sales'].reset_index(drop=True).iloc[:horizon]


        else:
            raise ValueError("additional preprocessing is required for the calendar")
    else:
        raise ValueError('Input train_df must contain exactly one item_id and one store_id')

    return df

validation_forecasts = []
for (item_id, store_id), group in train_window.groupby(['item_id','store_id']):
    forecast_df = seasonal_naive_forecast(
        train_df=group, horizon=28,
        season_length=7)
    validation_forecasts.append(forecast_df)
forecasts_df = pd.concat(validation_forecasts, ignore_index=True)


baseline_forecast = forecasts_df.merge(
    validation_df[['date', 'item_id','dept_id','cat_id','store_id', 'sales', 'sell_price', 'days_after_release']],
    how='left', left_on=['item_id','store_id','forecast_dates'], right_on = ['item_id','store_id','date'])
print(baseline_forecast[['item_id','store_id','date','sales','forecast_dates','forecast_sales']].head(10))
if baseline_forecast['forecast_sales'].isna().sum() == 0:
        print('OK with NA')
else: print( 'Please Check NA values in forecast sales')

mae = (baseline_forecast['sales'] - baseline_forecast['forecast_sales']).abs().mean()
print('MAE of baseline = ', round(mae,2))
rmse = (((baseline_forecast['sales'] - baseline_forecast['forecast_sales'])**2).mean())**0.5
print('RMSE of baseline = ', round(rmse,2))

       item_id store_id       date  sales forecast_dates forecast_sales
0  FOODS_1_001     CA_1 2016-03-01    0.0     2016-03-01            1.0
1  FOODS_1_001     CA_1 2016-03-02    1.0     2016-03-02            0.0
2  FOODS_1_001     CA_1 2016-03-03    0.0     2016-03-03            0.0
3  FOODS_1_001     CA_1 2016-03-04    0.0     2016-03-04            0.0
4  FOODS_1_001     CA_1 2016-03-05    2.0     2016-03-05            0.0
5  FOODS_1_001     CA_1 2016-03-06    0.0     2016-03-06            0.0
6  FOODS_1_001     CA_1 2016-03-07    0.0     2016-03-07            1.0
7  FOODS_1_001     CA_1 2016-03-08    0.0     2016-03-08            1.0
8  FOODS_1_001     CA_1 2016-03-09    0.0     2016-03-09            0.0
9  FOODS_1_001     CA_1 2016-03-10    1.0     2016-03-10            0.0
OK with NA
MAE of baseline =  1.21
RMSE of baseline =  2.76


In [6]:
baseline_forecast.columns

Index(['forecast_dates', 'item_id', 'store_id', 'forecast_sales', 'date',
       'dept_id', 'cat_id', 'sales', 'sell_price', 'days_after_release'],
      dtype='str')

In [15]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'browser'

# ============================================================
# 1. TOP-10 item_id по средней цене
# ============================================================

top10_items = (
    baseline_forecast.assign(
        pct_rev=lambda x: (
            (pd.to_numeric(x['sell_price'])*
             pd.to_numeric(x['sales']))
        )
    )
    .groupby('item_id')['pct_rev']
    .sum()
    .nlargest(10)
    .index
)

# ============================================================
# 2. Данные только для этих item_id
# ============================================================

dashboard_df = baseline_forecast[
    baseline_forecast['item_id'].isin(top10_items)
].copy()


# ============================================================
# 3. Forecast - Sales
# ============================================================
dashboard_df['forecast_minus_sales'] = (
    dashboard_df['forecast_sales'] - dashboard_df['sales']
)

# ============================================================
# 4. Создаём figure
# ============================================================

fig = go.Figure()

trace_items = []


# ============================================================
# 5. Добавляем trace для каждого item_id × store_id
# ============================================================

for item_id in sorted(top10_items):

    item_data = dashboard_df[
        dashboard_df['item_id'] == item_id
    ]

    for store_id, store_data in item_data.groupby('store_id'):

        store_data = store_data.sort_values('date')

        fig.add_trace(
            go.Scatter(
                x=store_data['date'],
                y=store_data['forecast_minus_sales'],
                mode='lines+markers',
                name=f'Store {store_id}',
                visible=False
            )
        )

        # Запоминаем item_id этого trace
        trace_items.append(item_id)


# ============================================================
# 6. Первый item_id показываем сразу
# ============================================================

first_item = sorted(top10_items)[0]

for i, item_id in enumerate(trace_items):

    if item_id == first_item:
        fig.data[i].visible = True


# ============================================================
# 7. Dropdown
# ============================================================

buttons = []

for selected_item in sorted(top10_items):

    visibility = [
        item_id == selected_item
        for item_id in trace_items
    ]

    buttons.append(
        dict(
            label=str(selected_item),
            method='update',
            args=[
                {'visible': visibility},
                {
                    'title': (
                        f'Item {selected_item}: '
                        'Forecast − Sales'
                    )
                }
            ]
        )
    )


# ============================================================
# 8. Layout
# ============================================================

fig.update_layout(
    title=f'Item {first_item}: Forecast − Sales',

    xaxis_title='Date',

    yaxis_title='Forecast − Sales',

    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.0,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    ],

    hovermode='x unified',

    legend_title='Store ID',

    template='plotly_white',

    height=600
)


# ============================================================
# 9. Показываем
# ============================================================

fig.show()


In [9]:
x = 123
x

123